In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Load data
df=pd.read_csv("../Dataset/customer_churn_prediction_dataset.csv")

In [26]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0001-XXXX,Male,0,Yes,Yes,5,No,Yes,No,No,...,No,No,No,No internet service,Month-to-month,Yes,Credit card,27.43,137.15,Yes
1,0002-XXXX,Female,0,No,No,42,Yes,No phone service,DSL,Yes,...,No,No,No internet service,No internet service,Two year,No,Electronic check,38.28,1607.76,Yes
2,0003-XXXX,Male,0,No,No,61,Yes,No phone service,No,No,...,No,Yes,Yes,No,One year,No,Bank transfer,106.44,6492.84,Yes
3,0004-XXXX,Male,1,No,Yes,22,Yes,No,No,No internet service,...,Yes,No,No internet service,Yes,Month-to-month,No,Electronic check,92.49,2034.78,No
4,0005-XXXX,Male,1,Yes,Yes,21,No,No,DSL,No,...,No internet service,No,No,No internet service,One year,No,Bank transfer,19.63,412.23,Yes


Create binary target (1 = churned, 0 = retained)

In [27]:
df['churn_flag']=(df['Churn']=='Yes').astype(int)

 Drop non-informative / leakage columns
 customerID is an identifier
 
 Churn is the original target label (replaced by churn_flag)

In [29]:
df.drop(columns=['customerID','Churn'],inplace=True)

In [30]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,churn_flag
0,Male,0,Yes,Yes,5,No,Yes,No,No,Yes,No,No,No,No internet service,Month-to-month,Yes,Credit card,27.43,137.15,1
1,Female,0,No,No,42,Yes,No phone service,DSL,Yes,Yes,No,No,No internet service,No internet service,Two year,No,Electronic check,38.28,1607.76,1
2,Male,0,No,No,61,Yes,No phone service,No,No,Yes,No,Yes,Yes,No,One year,No,Bank transfer,106.44,6492.84,1
3,Male,1,No,Yes,22,Yes,No,No,No internet service,No internet service,Yes,No,No internet service,Yes,Month-to-month,No,Electronic check,92.49,2034.78,0
4,Male,1,Yes,Yes,21,No,No,DSL,No,No,No internet service,No,No,No internet service,One year,No,Bank transfer,19.63,412.23,1


Tenure bucketing (lifecycle segmentation)

 Helps capture non-linear churn patterns across customer maturity stages

In [31]:
df['tenure_bucket']=pd.cut(
    df['tenure'],
    bins=[0,6,12,24,48,max(df['tenure'])],
    labels=['0-6','6-12','12-24','24-48','49+'],
    include_lowest=True
)

In [32]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,churn_flag,tenure_bucket
0,Male,0,Yes,Yes,5,No,Yes,No,No,Yes,...,No,No,No internet service,Month-to-month,Yes,Credit card,27.43,137.15,1,0-6
1,Female,0,No,No,42,Yes,No phone service,DSL,Yes,Yes,...,No,No internet service,No internet service,Two year,No,Electronic check,38.28,1607.76,1,24-48
2,Male,0,No,No,61,Yes,No phone service,No,No,Yes,...,Yes,Yes,No,One year,No,Bank transfer,106.44,6492.84,1,49+
3,Male,1,No,Yes,22,Yes,No,No,No internet service,No internet service,...,No,No internet service,Yes,Month-to-month,No,Electronic check,92.49,2034.78,0,12-24
4,Male,1,Yes,Yes,21,No,No,DSL,No,No,...,No,No,No internet service,One year,No,Bank transfer,19.63,412.23,1,12-24


 Drop TotalCharges to reduce redundancy
 
 TotalCharges is strongly related to tenure and monthly charges

In [33]:
df.drop(columns='TotalCharges',inplace=True)

In [34]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,churn_flag,tenure_bucket
0,Male,0,Yes,Yes,5,No,Yes,No,No,Yes,No,No,No,No internet service,Month-to-month,Yes,Credit card,27.43,1,0-6
1,Female,0,No,No,42,Yes,No phone service,DSL,Yes,Yes,No,No,No internet service,No internet service,Two year,No,Electronic check,38.28,1,24-48
2,Male,0,No,No,61,Yes,No phone service,No,No,Yes,No,Yes,Yes,No,One year,No,Bank transfer,106.44,1,49+
3,Male,1,No,Yes,22,Yes,No,No,No internet service,No internet service,Yes,No,No internet service,Yes,Month-to-month,No,Electronic check,92.49,0,12-24
4,Male,1,Yes,Yes,21,No,No,DSL,No,No,No internet service,No,No,No internet service,One year,No,Bank transfer,19.63,1,12-24


Split features (X) and target (y)

In [21]:
X=df.drop(columns=['churn_flag'])
y=df['churn_flag']

One-hot encode categorical features

drop_first=True prevents multicollinearity for linear models

In [ ]:
X=pd.get_dummies(X, drop_first=True)

In [35]:
print("X shape:", X.shape)
print("y distribution:\n", y.value_counts(normalize=True))
print("Missing values in X:", X.isna().sum().sum())

X shape: (300, 33)
y distribution:
 churn_flag
0    0.536667
1    0.463333
Name: proportion, dtype: float64
Missing values in X: 0
